# Phase 7 · Submission S4 — Knowledge Distillation: S1 retriever (12-layer A + LORMA) → 6-layer A student + LORMA

**Task:** Prior Case Retrieval. For every query case, rank the candidates of the same split's pool and return exactly 7 unique candidate ids, most relevant first. **Primary metric:** Micro-F1@7 (secondary Micro-P@7, Micro-R@7).

**Method.** The retrieval pipeline is the same as S1–S3. The model is a smaller student that learns from the S1 retriever.
1. **Teacher:** S1, the full 12-layer Phase-1 A with LORMA (126.7M parameters; dev Micro-F1@7 0.269). Its final case vectors for every train, dev and test document are read from **`Phase7_S1_Dataset`** (`outputs/teacher_vectors.pt`).
2. **Student:**
   - Phase-1 Architecture A with **every other layer kept** (layers 0, 2, 4, 6, 8, 10, as in DistilBERT): 6 layers, ≈ 82.6M parameters, ≈ 65% of the teacher.
   - The token and position embeddings and the final norm are copied from Phase 1, and the student is adapted with LORMA (r 8, α 16, `B = 0` at the start).
   - Every experiment still starts from the Phase-1 weights. The teacher only supplies training signals.
3. **What the student learns from the teacher:**
   - **Similarity scores (listwise KD):** for every query in a batch, the teacher's similarity distribution over all batch documents is matched with KL divergence at temperature 2: `T² · KL(softmax(teacher/T) ‖ softmax(student/T))`. Self and other relevant cases are masked exactly as in the contrastive loss.
   - **Hard negatives:** per (query, relevant case) pair, one hard negative comes from the TF-IDF top 50 and one from the **teacher's** top 50 non-relevant cases. The teacher-mined negatives are the cases the teacher thinks are similar but are not relevant.
   - **Total loss** = InfoNCE on the gold labels + 1.0 × KD.
4. **Input, case vector, LP-FT and selection are identical to S1:**
   - first + last chunks;
   - [mean ; final state] → standardised → 256-d projection → normalised mean over chunks;
   - Step 1 is the projection on frozen student features, Step 2 is LORMA + projection;
   - the best epoch on dev Micro-F1@7 is kept, and the query is always removed from its own ranking.
5. **Did KD keep the quality? Size, memory and speed:**
   - The teacher (rebuilt from S1's `best.pt`) and the student encode the same 100 dev documents on one GPU, for time and peak memory.
   - Also reported: parameters, dev Micro-F1@7 of both, and their top-7 agreement.
6. **No leakage:** train labels for training, dev labels for selection and analysis. TF-IDF is used only for hard negatives and as a reference.

**Before running (Kaggle)**
- Accelerator: **GPU T4 ×2**. Add-ons → Secrets: **`HF`**. Internet on.
- Add Input → datasets:
  - **`rohitbatra2004/phase111`** (Architecture A);
  - **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer);
  - **`Phase7_S1_Dataset`**, the output of P7_S1_A_LORMA: the teacher vectors and checkpoint.
- **Expected time ≈ 2–2.5 hours.** Checkpoints are saved every 300 steps and every epoch, and tokens and features are cached. To resume, add this notebook's previous output as an input and run again.

Outputs: `/kaggle/working/P7_S4_KD_S1Teacher_A6LayerStudent/`. Upload `submission.zip`, and save the output folder as the Kaggle dataset **`Phase7_S4_Dataset`**.

In [1]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile, re, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from sklearn.feature_extraction.text import TfidfVectorizer
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

PyTorch 2.11.0+cu128 | device: cuda | GPUs visible: 2


In [2]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P7_S4_KD_S1Teacher_A6LayerStudent"
TEACHER_RUN = "P7_S1_A_LORMA"              # the S1 output dataset (Phase7_S1_Dataset) holds the teacher
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"
K = 7                                       # answers per query (fixed by the task)

# Long documents: chunks of 510 BPE tokens (+ [BOS], [EOS] = 512). A setting = (which chunks, how many).
CHUNK_TOKENS = 510
KEEP_CHUNKS = 8                             # the first 8 and last 8 chunks of every case are tokenised and kept
CACHE_SETTING = ("firstlast", 16)           # frozen features are computed for these chunks (analysis of other settings)
EVAL_SETTING = ("firstlast", 8)             # first 4 + last 4 chunks: case vectors for selection, analysis and test
TRAIN_SETTING = ("firstlast", 4)            # first 2 + last 2 chunks per case while fine-tuning (memory)
SECTION_SETTINGS = (("first", 1), ("first", 4), ("last", 4), ("firstlast", 8), ("firstlast", 16))

# Student: Phase-1 A with every other layer kept (DistilBERT-style), adapted with LORMA on attention + FFN.
STUDENT_LAYERS = (0, 2, 4, 6, 8, 10)
LORMA_RANK = 8
LORMA_ALPHA = 16.0
LORMA_TARGETS_ATTN = ("q_proj", "k_proj", "v_proj", "out_proj")
LORMA_TARGETS_FFN = ("fc1", "fc2")

# Bi-encoder head and contrastive loss.
PROJ_DIM = 256
FEATURE_CLAMP = 10.0                        # standardised chunk features clipped to ±10
TEMPERATURE = 0.05
HARD_POOL = 50                              # hard negatives are sampled from the 50 most TF-IDF-similar non-relevant cases
HARD_PER_PAIR = 2                           # hard negatives per pair: 1 from TF-IDF + 1 mined by the teacher

# Knowledge distillation: listwise KL between teacher and student similarity distributions over the batch documents.
KD_WEIGHT = 1.0
KD_TEMPERATURE = 2.0
COST_DOCS = 100                             # dev documents encoded by teacher and student to compare time and memory

# Step 1 — projection trained on cached frozen features.
PROBE_EPOCHS = 30                           # evaluated every 3 epochs; the best on dev is kept
PROBE_LR = 1e-3
PROBE_PAIRS = 64                            # (query, relevant case) pairs per step

# Step 2 — student LORMA + projection fine-tuning.
EPOCHS = 3
PAIRS_PER_STEP = 4                          # 4 pairs × (query + relevant + 2 hard negatives) × 4 chunks = 64 chunks / step
LR_ADAPTER = 1e-4
LR_HEAD = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
GRAD_CHECKPOINTING = True                   # recompute block activations in the backward pass (memory)
SAVE_EVERY_STEPS = 300
CHECK_EVERY_STEPS = 600                     # quick dev evaluation during an epoch
EVAL_CHUNK_BATCH = 48                       # chunks per forward pass when encoding documents
USE_AMP = DEVICE.type == "cuda"
print("Run:", RUN_NAME, "| outputs:", WORK_DIR)

Run: P7_S4_KD_S1Teacher_A6LayerStudent | outputs: /kaggle/working/P7_S4_KD_S1Teacher_A6LayerStudent


In [3]:
# Cell 3 — Download Phase_7_PCR from Hugging Face using the Kaggle secret "HF".
from kaggle_secrets import UserSecretsClient
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_7_PCR/**"])
P7_DIR = DATA_DIR / "Phase_7_PCR"
FILES = {"train": ("Train/train.json", "Train/train_candidates.json"), "dev": ("Dev/dev.json", "Dev/dev_candidates.json"),
         "test": ("Test/test.json", "Test/test_candidates.json")}
for q_file, c_file in FILES.values():
    assert (P7_DIR / q_file).is_file() and (P7_DIR / c_file).is_file(), f"Missing {q_file} or {c_file}"
print("Phase 7 data ready in", P7_DIR)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Phase 7 data ready in /kaggle/working/data/Phase_7_PCR


In [4]:
# Cell 4 — Load queries and candidate pools, replace the masked placeholders, and register one document per text.
# Queries are also in their own pool (data analysis); a query is encoded once and is never ranked for itself.
PLACEHOLDERS = ((re.compile(r"<\s*citation\s*>", re.I), " Citation "), (re.compile(r"<\s*name\s*>", re.I), " Name "),
                (re.compile(r"<\s*org\s*>", re.I), " Org "))

def normalise(sentences):
    text = " ".join(sentences) if isinstance(sentences, list) else str(sentences)
    for pattern, replacement in PLACEHOLDERS:
        text = pattern.sub(replacement, text)
    return " ".join(text.split())

QUERIES, POOL_IDS, DOC_TEXT, DOC_INDEX = {}, {}, {}, {}
for split, (q_file, c_file) in FILES.items():
    queries, pool = json.load(open(P7_DIR / q_file)), json.load(open(P7_DIR / c_file))
    docs = {}                                                   # document key -> text (pool first, then queries not in it)
    for cid, rec in pool.items():
        docs[str(cid)] = normalise(rec["text"] if isinstance(rec, dict) else rec)
    QUERIES[split] = {}
    for qid, rec in queries.items():
        qid, text = str(qid), normalise(rec["text"])
        key = qid if docs.get(qid) == text else f"Q:{qid}"      # same id and text as a pool document -> share it
        docs.setdefault(key, text)
        QUERIES[split][qid] = {"key": key, "rel": [str(c) for c in rec.get("relevant_candidates", [])]}
    POOL_IDS[split] = [str(c) for c in pool]
    DOC_TEXT[split] = list(docs.values())
    DOC_INDEX[split] = {k: i for i, k in enumerate(docs)}
    missing = sum(c not in DOC_INDEX[split] for v in QUERIES[split].values() for c in v["rel"])
    assert missing == 0, f"{split}: {missing} relevant ids are not in the pool"
    shared = sum(v["key"] == q for q, v in QUERIES[split].items())
    print(f"{split}: {len(QUERIES[split])} queries ({shared} also in the pool) | pool {len(POOL_IDS[split]):,} | "
          f"documents to encode {len(DOC_TEXT[split]):,} | labelled queries {sum(bool(v['rel']) for v in QUERIES[split].values())}")
TRAIN_QUERIES = [q for q, v in QUERIES["train"].items() if v["rel"]]
DEV_QUERIES = [q for q, v in QUERIES["dev"].items() if v["rel"]]
TEST_QUERIES = list(QUERIES["test"])
print("Example after normalisation:", DOC_TEXT["dev"][0][:300])

train: 827 queries (827 also in the pool) | pool 4,320 | documents to encode 4,320 | labelled queries 817
dev: 118 queries (118 also in the pool) | pool 1,023 | documents to encode 1,023 | labelled queries 118
test: 237 queries (237 also in the pool) | pool 1,727 | documents to encode 1,727 | labelled queries 0
Example after normalisation: Name , J. 1. This is an appeal from the decision of Org at Fort William in Bengal, dated November 25, 1924, by which the judgment of the Subordinate Judge of Asansol in Zillah Burdwan, dated February 5, 1923, was reversed and the suit dismissed with costs. The appellant prays that the decision of Or


In [5]:
# Cell 5 — Tokenise every document once (cased Phase-1 BPE) and keep its first and last KEEP_CHUNKS chunks (cached);
# chunk selection rules: "first" k, "last" k, "firstlast" k = first k/2 + last k/2 (all chunks if the case is shorter).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]"))
assert (PAD_ID, BOS_ID, EOS_ID) == (0, 2, 3)

def tokenise_split(texts):
    """For each document: {'n': total chunks, 'chunks': {chunk number: uint16 token array}} (first and last KEEP_CHUNKS)."""
    out = []
    for start in range(0, len(texts), 64):
        for enc in tokenizer.encode_batch(texts[start:start + 64]):
            ids = np.array(enc.ids, dtype=np.uint16)
            n = max(1, -(-len(ids) // CHUNK_TOKENS))
            keep = sorted(set(range(min(KEEP_CHUNKS, n))) | set(range(max(0, n - KEEP_CHUNKS), n)))
            out.append({"n": n, "chunks": {c: ids[c * CHUNK_TOKENS:(c + 1) * CHUNK_TOKENS] for c in keep}})
    return out

cache = WORK_DIR / "tokens_cache.pt"
if not cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/tokens_cache.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], cache)
if cache.exists():
    TOKENS = torch.load(cache, weights_only=False)
    print("Loaded cached tokens from", cache)
else:
    start_time = time.time()
    TOKENS = {split: tokenise_split(DOC_TEXT[split]) for split in DOC_TEXT}
    torch.save(TOKENS, cache)
    print(f"Tokenised {sum(len(v) for v in TOKENS.values()):,} documents in {(time.time() - start_time) / 60:.1f} min → {cache}")
for split in DOC_TEXT:
    assert len(TOKENS[split]) == len(DOC_TEXT[split])

def chosen(chunk_no, n, setting):
    """Is chunk `chunk_no` of a case with `n` chunks part of `setting`?"""
    kind, k = setting
    if kind == "first":
        return chunk_no < k
    if kind == "last":
        return chunk_no >= n - k
    return chunk_no < k // 2 or chunk_no >= n - k // 2           # "firstlast"

def doc_chunks(split, d, setting):
    """Token lists ([BOS] … [EOS]) of the chunks a setting uses for one document, in document order."""
    t = TOKENS[split][d]
    return [[BOS_ID] + t["chunks"][c].tolist() + [EOS_ID] for c in sorted(t["chunks"]) if chosen(c, t["n"], setting)]

for split in DOC_TEXT:
    n = np.array([t["n"] for t in TOKENS[split]])
    print(f"{split}: chunks per document median {int(np.median(n))}, p90 {int(np.percentile(n, 90))}, max {n.max()} | "
          f"fully covered by {EVAL_SETTING[1]} chunks: {100 * np.mean(n <= EVAL_SETTING[1]):.1f}%")

Tokenised 7,070 documents in 1.3 min → /kaggle/working/P7_S4_KD_S1Teacher_A6LayerStudent/tokens_cache.pt
train: chunks per document median 9, p90 29, max 949 | fully covered by 8 chunks: 47.5%
dev: chunks per document median 10, p90 39, max 949 | fully covered by 8 chunks: 42.4%
test: chunks per document median 10, p90 32, max 949 | fully covered by 8 chunks: 44.1%


In [6]:
# Cell 6 — TF-IDF over each split's pool: hard negatives for every train query (top HARD_POOL non-relevant, never the query
# itself) and the TF-IDF reference score on dev. Neither is ever part of a submitted ranking.
def micro_at_k(ranked, gold):
    """Official metric: micro precision / recall / F1 over the first K answers; plus Recall@20/50/100."""
    hits = sum(len(set(r[:K]) & set(g)) for r, g in zip(ranked, gold))
    n_gold = sum(len(g) for g in gold)
    p, r = hits / (K * len(gold)), hits / max(n_gold, 1)
    out = {"micro_p@7": round(p, 4), "micro_r@7": round(r, 4), "micro_f1@7": round(2 * p * r / max(p + r, 1e-9), 4)}
    out |= {f"recall@{k}": round(sum(len(set(x[:k]) & set(g)) for x, g in zip(ranked, gold)) / max(n_gold, 1), 4) for k in (20, 50, 100)}
    return out

def rank_queries(scores, split, query_ids, depth=100):
    """scores [queries, documents] → for each query the top `depth` pool ids, without the query itself."""
    pool_rows = torch.tensor([DOC_INDEX[split][c] for c in POOL_IDS[split]])
    s = scores[:, pool_rows].clone().float()
    for i, q in enumerate(query_ids):
        own = QUERIES[split][q]["key"]
        if own in POOL_IDS[split]:
            s[i, POOL_IDS[split].index(own)] = -float("inf")
        if q in POOL_IDS[split]:
            s[i, POOL_IDS[split].index(q)] = -float("inf")
    top = s.topk(min(depth, s.size(1)), dim=1).indices.tolist()
    return [[POOL_IDS[split][j] for j in row] for row in top]

def gold(split, query_ids):
    return [QUERIES[split][q]["rel"] for q in query_ids]

def tfidf_scores(split, query_ids):
    vec = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z][a-z0-9]+\b", stop_words="english", min_df=2,
                          max_df=0.5, sublinear_tf=True, max_features=200_000, dtype=np.float32)
    D = vec.fit_transform(DOC_TEXT[split])
    Q = D[[DOC_INDEX[split][QUERIES[split][q]["key"]] for q in query_ids]]
    return torch.from_numpy((Q @ D.T).toarray())

hard_cache = WORK_DIR / "hard_negatives.pt"
if not hard_cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/hard_negatives.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], hard_cache)
if hard_cache.exists():
    HARD = torch.load(hard_cache, weights_only=False)
    print("Loaded cached hard negatives and TF-IDF reference from", hard_cache)
else:
    start_time = time.time()
    ranked = rank_queries(tfidf_scores("train", TRAIN_QUERIES), "train", TRAIN_QUERIES, depth=HARD_POOL + 60)
    negatives = {q: [c for c in r if c not in set(QUERIES["train"][q]["rel"])][:HARD_POOL] for q, r in zip(TRAIN_QUERIES, ranked)}
    tfidf_dev = micro_at_k(rank_queries(tfidf_scores("dev", DEV_QUERIES), "dev", DEV_QUERIES), gold("dev", DEV_QUERIES))
    tfidf_dev_ranked = rank_queries(tfidf_scores("dev", DEV_QUERIES), "dev", DEV_QUERIES, depth=K)
    HARD = {"negatives": negatives, "tfidf_dev": tfidf_dev, "tfidf_dev_top7": dict(zip(DEV_QUERIES, tfidf_dev_ranked))}
    torch.save(HARD, hard_cache)
    print(f"TF-IDF hard negatives for {len(negatives)} train queries in {time.time() - start_time:.0f} s")
print("TF-IDF reference on dev (not submitted):", HARD["tfidf_dev"])

TF-IDF hard negatives for 817 train queries in 21 s
TF-IDF reference on dev (not submitted): {'micro_p@7': 0.4007, 'micro_r@7': 0.3402, 'micro_f1@7': 0.368, 'recall@20': 0.4728, 'recall@50': 0.5827, 'recall@100': 0.668}


In [7]:
# Cell 7 — Phase-1 Architecture A classes and weights (verified against the metadata), and the 6-layer student state:
# Phase-1 blocks STUDENT_LAYERS renumbered 0…5, with the Phase-1 embeddings and final norm.
a_paths = sorted(glob.glob("/kaggle/input/**/architecture_A/model_A_state_dict.pt", recursive=True))
assert a_paths, "Add the dataset rohitbatra2004/phase111 as an input."
A_ROOT = Path(a_paths[0]).parent
A_METADATA = json.load(open(A_ROOT / "model_A_metadata.json"))
A_CONFIG = A_METADATA["model_config"]
assert A_CONFIG["pre_norm"] and A_CONFIG["tie_embeddings"], "Phase-1 A is a pre-LN decoder with tied embeddings"

class CausalSelfAttention(nn.Module):
    """Same parameters as Phase 1; uses PyTorch's fused causal attention for speed and memory."""
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.n_heads, self.dropout = cfg["n_heads"], cfg["dropout"]
        self.q_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.k_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.v_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.out_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.resid_dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, T, C = x.shape
        split = lambda t: t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        out = F.scaled_dot_product_attention(split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x)),
                                             dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.resid_dropout(self.out_proj(out.transpose(1, 2).reshape(B, T, C)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["use_bias"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["use_bias"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["d_model"])
        self.attention = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["d_model"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        return x + self.feed_forward(self.ln_2(x))

class ArchitectureABackbone(nn.Module):
    """Phase-1 decoder-only LM without the vocabulary projection.
    Returns the final-normalised state of every token: [batch, tokens, d_model]."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(cfg["d_model"])
        # Tied output weight, kept so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_weight = self.token_embedding.weight
        self.grad_checkpointing = False

    def forward(self, input_ids):
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))
        for block in self.blocks:
            if self.grad_checkpointing and torch.is_grad_enabled():
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.final_norm(x)

saved = torch.load(a_paths[0], map_location="cpu", weights_only=True)
# The export is either the raw state dict or a dict that wraps it under "state_dict".
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved

check = ArchitectureABackbone(A_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(A_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(A_CONFIG, indent=2))
print(f"Phase-1 Architecture A weights verified from {A_ROOT} — {n_backbone:,} parameters")

STUDENT_CONFIG = dict(A_CONFIG, n_layers=len(STUDENT_LAYERS))
STUDENT_STATE = {}
for name, tensor in PHASE1_STATE.items():
    m = re.match(r"blocks\.(\d+)\.(.*)", name)
    if m is None:
        STUDENT_STATE[name] = tensor
    elif int(m.group(1)) in STUDENT_LAYERS:
        STUDENT_STATE[f"blocks.{STUDENT_LAYERS.index(int(m.group(1)))}.{m.group(2)}"] = tensor
check = ArchitectureABackbone(STUDENT_CONFIG)
check.load_state_dict(STUDENT_STATE, strict=True)
n_student_backbone = sum(p.numel() for p in check.parameters())
del check
print(f"Student backbone: Phase-1 layers {STUDENT_LAYERS} → {n_student_backbone:,} parameters "
      f"({100 * n_student_backbone / n_backbone:.1f}% of the 12-layer model)")

{
  "vocab_size": 50000,
  "context_length": 2048,
  "d_model": 768,
  "n_layers": 12,
  "n_heads": 12,
  "d_ff": 3072,
  "dropout": 0.1,
  "activation": "gelu",
  "norm_type": "layernorm",
  "pre_norm": true,
  "position_type": "learned",
  "tie_embeddings": true,
  "use_bias": true
}
Phase-1 Architecture A weights verified from /kaggle/input/datasets/rohitbatra2004/phase111/architecture_A — 125,028,864 parameters
Student backbone: Phase-1 layers (0, 2, 4, 6, 8, 10) → 82,501,632 parameters (66.0% of the 12-layer model)


In [8]:
# Cell 8 — LORMA: multiplicative low-rank update of a frozen linear layer, W_eff = (I + (α/r)·B·A)·W0.
# Applied to q/k/v/out and fc1/fc2 in all 12 layers; B = 0 at start, so the model initially equals Phase 1.
class LoRMALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lorma_A = nn.Parameter(torch.empty(rank, base.out_features))
        self.lorma_B = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

    def forward(self, x):
        w0x = F.linear(x, self.base.weight)                               # W0 x (without bias)
        update = F.linear(F.linear(w0x, self.lorma_A), self.lorma_B)       # B (A (W0 x))
        out = w0x + self.scale * update
        return out + self.base.bias if self.base.bias is not None else out

def make_lorma_backbone(cfg=STUDENT_CONFIG, state=STUDENT_STATE):
    bb = ArchitectureABackbone(cfg)
    bb.load_state_dict(state, strict=True)
    for p in bb.parameters():
        p.requires_grad_(False)
    for block in bb.blocks:
        for name in LORMA_TARGETS_ATTN:
            setattr(block.attention, name, LoRMALinear(getattr(block.attention, name), LORMA_RANK, LORMA_ALPHA))
        for name in LORMA_TARGETS_FFN:
            setattr(block.feed_forward, name, LoRMALinear(getattr(block.feed_forward, name), LORMA_RANK, LORMA_ALPHA))
    return bb

# Checks on the device (student): (1) with B = 0 the output equals the frozen 6-layer Phase-1 student; (2) A and B
# receive gradients, also through gradient checkpointing.
probe_ids = torch.randint(4, 1000, (2, 16), device=DEVICE)
plain = ArchitectureABackbone(STUDENT_CONFIG); plain.load_state_dict(STUDENT_STATE, strict=True); plain = plain.to(DEVICE).eval()
probe_bb = make_lorma_backbone().to(DEVICE).eval()
with torch.no_grad():
    assert torch.allclose(plain(probe_ids), probe_bb(probe_ids), atol=1e-4), "LORMA with B = 0 must equal the Phase-1 student"
    lorma_layers = [m for m in probe_bb.modules() if isinstance(m, LoRMALinear)]
    for m in lorma_layers:
        m.lorma_B.normal_(0, 1e-3)                                       # temporary non-zero B for the gradient check
probe_bb.grad_checkpointing = True
probe_bb(probe_ids).float().pow(2).mean().backward()
assert all(m.lorma_A.grad is not None and m.lorma_A.grad.abs().sum() > 0 for m in lorma_layers), "LORMA gets no gradient"
n_lorma = sum(p.numel() for p in probe_bb.parameters() if p.requires_grad)
print(f"LORMA on {len(lorma_layers)} layers — {n_lorma:,} trainable backbone parameters ({100 * n_lorma / n_student_backbone:.2f}% of the student)")
del plain, probe_bb, lorma_layers
torch.cuda.empty_cache()

LORMA on 36 layers — 663,552 trainable backbone parameters (0.80% of the student)


In [9]:
# Cell 9 — Bi-encoder. ChunkEncoder (split over both GPUs): chunk tokens → [mean ; final state] of the token states.
# Retriever: standardise (frozen-model statistics) → projection to PROJ_DIM → L2 norm → case vector = normalised mean.
class ChunkEncoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, ids):
        mask = ids.ne(PAD_ID)
        states = self.backbone(ids).float()
        m = mask.unsqueeze(-1).float()
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1.0)
        last = states[torch.arange(states.size(0), device=states.device), mask.sum(1) - 1]
        return torch.cat([mean, last], dim=-1)                          # [chunks, 2·d_model]

class Retriever(nn.Module):
    def __init__(self, backbone, hidden, dim):
        super().__init__()
        self.encoder = ChunkEncoder(backbone)
        self.register_buffer("feat_mean", torch.zeros(2 * hidden))
        self.register_buffer("feat_std", torch.ones(2 * hidden))
        self.proj = nn.Linear(2 * hidden, dim)
        nn.init.orthogonal_(self.proj.weight); nn.init.zeros_(self.proj.bias)

    def chunk_vectors(self, raw):
        x = ((raw - self.feat_mean) / self.feat_std).clamp(-FEATURE_CLAMP, FEATURE_CLAMP)
        return F.normalize(self.proj(x), dim=-1)

def pool(vectors, lengths, mode="mean"):
    """Chunk vectors (grouped by document, in order) → one L2-normalised vector per document."""
    lengths = torch.as_tensor(lengths, device=vectors.device)
    out = torch.segment_reduce(vectors, mode, lengths=lengths, axis=0)
    return F.normalize(out, dim=-1)

torch.manual_seed(SEED)
model = Retriever(make_lorma_backbone(), A_CONFIG["d_model"], PROJ_DIM).to(DEVICE)
N_GPUS = torch.cuda.device_count()
encoder = nn.DataParallel(model.encoder) if N_GPUS > 1 else model.encoder
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = TRAINABLE_NAMES + ["feat_mean", "feat_std"]
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%) | GPUs used: {max(N_GPUS, 1)}")

Trainable parameters: 1,057,024 of 83,558,656 (1.27%) | GPUs used: 2


In [10]:
# Cell 10 — Encoding helpers: padded chunk batches, raw chunk features of whole splits (frozen or adapted),
# and case vectors + dev evaluation (Micro-P/R/F1@7, query removed from its own ranking).
def chunk_batch(chunk_lists):
    T = max(len(x) for x in chunk_lists)
    ids = torch.full((len(chunk_lists), T), PAD_ID, dtype=torch.long)
    for i, x in enumerate(chunk_lists):
        ids[i, :len(x)] = torch.tensor(x)
    return ids.to(DEVICE)

@torch.no_grad()
def raw_features(split, setting, doc_ids=None):
    """Raw [mean ; final] features of every chosen chunk of the documents (in order) and the chunk count per document."""
    encoder.eval()
    doc_ids = range(len(TOKENS[split])) if doc_ids is None else doc_ids
    chunks, lengths, positions = [], [], []
    for d in doc_ids:
        cs = doc_chunks(split, d, setting)
        t = TOKENS[split][d]
        chunks += cs; lengths.append(len(cs))
        positions += [(c, t["n"]) for c in sorted(t["chunks"]) if chosen(c, t["n"], setting)]
    order = sorted(range(len(chunks)), key=lambda i: len(chunks[i]))       # similar lengths together (less padding)
    feats = torch.empty(len(chunks), 2 * A_CONFIG["d_model"])
    for s in range(0, len(order), EVAL_CHUNK_BATCH):
        idx = order[s:s + EVAL_CHUNK_BATCH]
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            feats[idx] = encoder(chunk_batch([chunks[i] for i in idx])).float().cpu()
    return feats, lengths, positions

@torch.no_grad()
def case_vectors(split, setting=EVAL_SETTING, mode="mean"):
    feats, lengths, _ = raw_features(split, setting)
    vecs = model.chunk_vectors(feats.to(DEVICE))
    return pool(vecs, lengths, mode)

def evaluate_vectors(vectors, split, query_ids, depth=100):
    rows = torch.tensor([DOC_INDEX[split][QUERIES[split][q]["key"]] for q in query_ids], device=vectors.device)
    scores = (vectors[rows] @ vectors.T).cpu()
    ranked = rank_queries(scores, split, query_ids, depth)
    return micro_at_k(ranked, gold(split, query_ids)), ranked

def describe(r):
    return f"Micro-F1@7 {r['micro_f1@7']:.4f} (P {r['micro_p@7']:.4f}, R {r['micro_r@7']:.4f}) | Recall@100 {r['recall@100']:.4f}"

def spread(vectors):
    """Mean cosine between random document pairs; close to 1 means every case looks the same (collapse)."""
    i = torch.randint(0, len(vectors), (2000,)); j = torch.randint(0, len(vectors), (2000,))
    return float((vectors[i] * vectors[j]).sum(-1).mean())
print("Encoding helpers ready")

Encoding helpers ready


In [11]:
# Cell 11 — Teacher (S1): load its case vectors for every document, check they reproduce S1's dev score, and mine the
# teacher's hard negatives (its top HARD_POOL non-relevant train cases per query, never the query itself).
found = sorted(glob.glob(f"/kaggle/input/**/{TEACHER_RUN}/outputs/teacher_vectors.pt", recursive=True))
assert found, "Add the S1 output dataset (Phase7_S1_Dataset) as an input."
TEACHER = torch.load(found[0], map_location="cpu", weights_only=False)
for split in ("train", "dev", "test"):
    assert TEACHER[split]["keys"] == list(DOC_INDEX[split]), f"teacher documents of {split} do not match this notebook"
T_VECS = {split: TEACHER[split]["vectors"].float().to(DEVICE) for split in ("train", "dev", "test")}
teacher_dev, teacher_dev_ranked = evaluate_vectors(T_VECS["dev"], "dev", DEV_QUERIES)
print(f"Teacher ({found[0]}): dev {describe(teacher_dev)} | setting {TEACHER['setting']} {TEACHER['pooling']}")
assert abs(teacher_dev["micro_f1@7"] - TEACHER["dev_result"]["micro_f1@7"]) < 0.01, "teacher vectors do not reproduce S1"
t_scores = torch.cat([T_VECS["train"][[DOC_INDEX["train"][QUERIES["train"][q]["key"]] for q in TRAIN_QUERIES[s:s + 256]]]
                      @ T_VECS["train"].T for s in range(0, len(TRAIN_QUERIES), 256)]).cpu()
t_ranked = rank_queries(t_scores, "train", TRAIN_QUERIES, depth=HARD_POOL + 60)
TEACHER_NEGATIVES = {q: [c for c in r if c not in set(QUERIES["train"][q]["rel"])][:HARD_POOL]
                     for q, r in zip(TRAIN_QUERIES, t_ranked)}
teacher_train = micro_at_k([r[:K] for r in t_ranked], gold("train", TRAIN_QUERIES))
shared = np.mean([len(set(TEACHER_NEGATIVES[q]) & set(HARD["negatives"][q])) / max(len(TEACHER_NEGATIVES[q]), 1) for q in TRAIN_QUERIES])
print(f"Teacher on train (it was trained on these): {describe(teacher_train)}")
print(f"Teacher-mined hard negatives for {len(TEACHER_NEGATIVES)} queries; {100 * shared:.0f}% also in the TF-IDF pool")
del t_scores

Teacher (/kaggle/input/datasets/rohitbatra2004/phase7-s1-dataset/P7_S1_A_LORMA/outputs/teacher_vectors.pt): dev Micro-F1@7 0.2690 (P 0.2930, R 0.2487) | Recall@100 0.6917 | setting ['firstlast', 16] mean
Teacher on train (it was trained on these): Micro-F1@7 0.2254 (P 0.2173, R 0.2342) | Recall@100 0.2342
Teacher-mined hard negatives for 817 queries; 31% also in the TF-IDF pool


In [12]:
# Cell 12 — Frozen Phase-1 features (LORMA B = 0) of the CACHE_SETTING chunks of every train / dev / test document, cached.
# Standardisation statistics come from the train chunks; they stay fixed for the rest of the run.
feature_cache = WORK_DIR / "frozen_features.pt"
if not feature_cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/frozen_features.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], feature_cache)
if feature_cache.exists():
    FROZEN = torch.load(feature_cache, weights_only=False)
    print("Loaded cached frozen features from", feature_cache)
else:
    start_time = time.time()
    FROZEN = {}
    for split in ("train", "dev", "test"):
        feats, lengths, positions = raw_features(split, CACHE_SETTING)
        FROZEN[split] = {"feats": feats.half(), "lengths": lengths, "positions": positions}
        print(f"  {split}: {len(feats):,} chunks of {len(lengths):,} documents | {(time.time() - start_time) / 60:.1f} min", flush=True)
    torch.save(FROZEN, feature_cache)
feat_mean = FROZEN["train"]["feats"].float().mean(0)
feat_std = FROZEN["train"]["feats"].float().std(0).clamp(min=1e-4)
with torch.no_grad():
    model.feat_mean.copy_(feat_mean); model.feat_std.copy_(feat_std)
print(f"Largest raw feature magnitude {FROZEN['train']['feats'].float().abs().max():.1f} vs mean |x| "
      f"{FROZEN['train']['feats'].float().abs().mean():.2f} → per-dimension standardisation")

def frozen_subset(split, setting):
    """Rows of the cached features that belong to `setting`, and the new chunk count per document."""
    f = FROZEN[split]
    keep = torch.tensor([chosen(c, n, setting) for c, n in f["positions"]])
    lengths, start = [], 0
    for L in f["lengths"]:
        lengths.append(int(keep[start:start + L].sum())); start += L
    return f["feats"][keep].float(), lengths

  train: 40,955 chunks of 4,320 documents | 2.3 min
  dev: 10,200 chunks of 1,023 documents | 2.9 min
  test: 17,111 chunks of 1,727 documents | 3.9 min
Largest raw feature magnitude 8.4 vs mean |x| 0.89 → per-dimension standardisation


In [13]:
# Cell 13 — How Phase-1 representations transfer without any training: frozen standardised features, cosine similarity,
# for each chunk selection and for mean vs max pooling over chunks (dev Micro-F1@7).
def standardised(feats):
    return ((feats - feat_mean) / feat_std).clamp(-FEATURE_CLAMP, FEATURE_CLAMP)

transfer = {}
for setting in SECTION_SETTINGS:
    feats, lengths = frozen_subset("dev", setting)
    x = F.normalize(standardised(feats), dim=-1).to(DEVICE)
    for mode in ("mean", "max"):
        result, _ = evaluate_vectors(pool(x, lengths, mode), "dev", DEV_QUERIES)
        transfer[f"{setting[0]}_{setting[1]}_{mode}"] = result
        print(f"  frozen Phase-1, {setting[0]:9s} {setting[1]:2d} chunks, {mode:4s} pooling: {describe(result)}")
print("TF-IDF reference:", describe(HARD["tfidf_dev"]))

  frozen Phase-1, first      1 chunks, mean pooling: Micro-F1@7 0.0989 (P 0.1077, R 0.0915) | Recall@100 0.4378
  frozen Phase-1, first      1 chunks, max  pooling: Micro-F1@7 0.0989 (P 0.1077, R 0.0915) | Recall@100 0.4378
  frozen Phase-1, first      4 chunks, mean pooling: Micro-F1@7 0.1790 (P 0.1949, R 0.1655) | Recall@100 0.5704
  frozen Phase-1, first      4 chunks, max  pooling: Micro-F1@7 0.1679 (P 0.1828, R 0.1552) | Recall@100 0.5498
  frozen Phase-1, last       4 chunks, mean pooling: Micro-F1@7 0.1601 (P 0.1743, R 0.1480) | Recall@100 0.5447
  frozen Phase-1, last       4 chunks, max  pooling: Micro-F1@7 0.1379 (P 0.1501, R 0.1274) | Recall@100 0.4861
  frozen Phase-1, firstlast  8 chunks, mean pooling: Micro-F1@7 0.1923 (P 0.2094, R 0.1778) | Recall@100 0.6136
  frozen Phase-1, firstlast  8 chunks, max  pooling: Micro-F1@7 0.1612 (P 0.1755, R 0.1490) | Recall@100 0.5108
  frozen Phase-1, firstlast 16 chunks, mean pooling: Micro-F1@7 0.2201 (P 0.2397, R 0.2035) | Recall@100

In [14]:
# Cell 14 — Loss = InfoNCE on gold labels + KD (listwise KL to the teacher's similarities over the batch documents);
# Step 1 (the "LP" of LP-FT): train only the projection on cached frozen student features; best on dev = "epoch 0".
TRAIN_PAIRS = [(q, c) for q in TRAIN_QUERIES for c in QUERIES["train"][q]["rel"]]
RELEVANT_KEYS = {q: {DOC_INDEX["train"][c] for c in QUERIES["train"][q]["rel"]} for q in TRAIN_QUERIES}
print(f"{len(TRAIN_PAIRS):,} (query, relevant case) training pairs from {len(TRAIN_QUERIES)} queries")

LOSS_PARTS = {"ce": 0.0, "kd": 0.0}
def contrastive_loss(doc_vecs, batch_docs, pairs):
    """doc_vecs [N, dim] for batch_docs (document indices); pairs = list of (query, relevant case).
    InfoNCE over every batch document except the query itself (its other relevant cases masked) + KD_WEIGHT × listwise
    KL between the teacher's and the student's similarity distributions over the same documents (same mask)."""
    position = {d: i for i, d in enumerate(batch_docs)}
    q_rows = [position[DOC_INDEX["train"][QUERIES["train"][q]["key"]]] for q, _ in pairs]
    targets = torch.tensor([position[DOC_INDEX["train"][c]] for _, c in pairs], device=doc_vecs.device)
    logits = doc_vecs[q_rows] @ doc_vecs.T / TEMPERATURE
    mask = torch.zeros_like(logits, dtype=torch.bool)
    for i, (q, c) in enumerate(pairs):
        mask[i, q_rows[i]] = True
        for d in RELEVANT_KEYS[q]:
            if d in position and position[d] != targets[i].item():
                mask[i, position[d]] = True
    ce = F.cross_entropy(logits.masked_fill(mask, -1e4), targets)
    t = T_VECS["train"][torch.tensor(batch_docs, device=DEVICE)]
    t_logits = (t[q_rows] @ t.T / TEMPERATURE).masked_fill(mask, -1e4)
    kd = F.kl_div(F.log_softmax(logits.masked_fill(mask, -1e4) / KD_TEMPERATURE, dim=-1),
                  F.softmax(t_logits / KD_TEMPERATURE, dim=-1), reduction="batchmean") * KD_TEMPERATURE ** 2
    LOSS_PARTS["ce"], LOSS_PARTS["kd"] = ce.item(), kd.item()
    return ce + KD_WEIGHT * kd

def pair_batch(pairs, rng):
    """Document indices for a batch: each pair's query, relevant case, one TF-IDF and one teacher-mined hard negative."""
    docs = []
    for q, c in pairs:
        negs = rng.sample(HARD["negatives"][q], 1) + rng.sample(TEACHER_NEGATIVES[q], HARD_PER_PAIR - 1)
        for d in [QUERIES["train"][q]["key"], c] + negs:
            i = DOC_INDEX["train"][d]
            if i not in docs:
                docs.append(i)
    return docs

train_feats, train_lengths = frozen_subset("train", EVAL_SETTING)
train_x = standardised(train_feats).to(DEVICE)
offsets = np.concatenate([[0], np.cumsum(train_lengths)])
dev_feats, dev_lengths = frozen_subset("dev", EVAL_SETTING)
dev_x = standardised(dev_feats).to(DEVICE)

def probe_doc_vectors(proj, docs):
    rows = torch.cat([torch.arange(offsets[d], offsets[d + 1]) for d in docs]).to(DEVICE)
    return pool(F.normalize(proj(train_x[rows]), dim=-1), [train_lengths[d] for d in docs])

LATEST, BEST = CKPT_DIR / "latest.pt", CKPT_DIR / "best.pt"
def trainable_state():
    full = model.state_dict()
    return {n: full[n].detach().cpu() for n in SAVED_NAMES}

torch.manual_seed(SEED)
proj = nn.Linear(2 * A_CONFIG["d_model"], PROJ_DIM).to(DEVICE)
nn.init.orthogonal_(proj.weight); nn.init.zeros_(proj.bias)
optimizer = torch.optim.AdamW(proj.parameters(), lr=PROBE_LR, weight_decay=0.01)
rng = random.Random(SEED)
best_probe, probe_history = (-1.0, None, 0), []
start_time = time.time()
for epoch in range(1, PROBE_EPOCHS + 1):
    order = TRAIN_PAIRS[:]; rng.shuffle(order)
    for s in range(0, len(order), PROBE_PAIRS):
        pairs = order[s:s + PROBE_PAIRS]
        docs = pair_batch(pairs, rng)
        loss = contrastive_loss(probe_doc_vectors(proj, docs), docs, pairs)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 3 == 0:
        with torch.no_grad():
            result, _ = evaluate_vectors(pool(F.normalize(proj(dev_x), dim=-1), dev_lengths), "dev", DEV_QUERIES)
        probe_history.append({"epoch": epoch, "loss": round(loss.item(), 4), **result})
        print(f"  probe epoch {epoch:2d}: last loss {loss.item():.3f} (InfoNCE {LOSS_PARTS['ce']:.3f}, KD {LOSS_PARTS['kd']:.3f}) | "
              f"dev {describe(result)}", flush=True)
        if result["micro_f1@7"] > best_probe[0]:
            best_probe = (result["micro_f1@7"], {k: v.detach().clone() for k, v in proj.state_dict().items()}, epoch)
PROBE_F1, probe_state, PROBE_EPOCH = best_probe
with torch.no_grad():                                      # copy the trained projection into the model
    model.proj.weight.copy_(probe_state["weight"]); model.proj.bias.copy_(probe_state["bias"])
probe_dev, _ = evaluate_vectors(case_vectors("dev"), "dev", DEV_QUERIES)
print(f"Projection (probe epoch {PROBE_EPOCH}) copied into the model → epoch 0: {describe(probe_dev)} | "
      f"{(time.time() - start_time) / 60:.1f} min")
assert abs(probe_dev["micro_f1@7"] - PROBE_F1) < 0.02, "The model does not reproduce the probe"
torch.save({"trainable": trainable_state(), "epoch": 0, "dev": probe_dev}, BEST)       # epoch 0 = projection only
del train_x, dev_x, proj
torch.cuda.empty_cache()

5,308 (query, relevant case) training pairs from 817 queries
  probe epoch  3: last loss 4.424 (InfoNCE 3.168, KD 1.256) | dev Micro-F1@7 0.2335 (P 0.2542, R 0.2158) | Recall@100 0.6588
  probe epoch  6: last loss 3.662 (InfoNCE 2.732, KD 0.930) | dev Micro-F1@7 0.2657 (P 0.2893, R 0.2456) | Recall@100 0.6588
  probe epoch  9: last loss 3.043 (InfoNCE 2.289, KD 0.753) | dev Micro-F1@7 0.2601 (P 0.2833, R 0.2405) | Recall@100 0.6619
  probe epoch 12: last loss 3.072 (InfoNCE 2.350, KD 0.722) | dev Micro-F1@7 0.2601 (P 0.2833, R 0.2405) | Recall@100 0.6721
  probe epoch 15: last loss 2.972 (InfoNCE 2.211, KD 0.760) | dev Micro-F1@7 0.2524 (P 0.2748, R 0.2333) | Recall@100 0.6752
  probe epoch 18: last loss 3.022 (InfoNCE 2.233, KD 0.789) | dev Micro-F1@7 0.2657 (P 0.2893, R 0.2456) | Recall@100 0.6732
  probe epoch 21: last loss 2.478 (InfoNCE 1.824, KD 0.654) | dev Micro-F1@7 0.2646 (P 0.2881, R 0.2446) | Recall@100 0.6783
  probe epoch 24: last loss 2.637 (InfoNCE 1.888, KD 0.749) | de

In [15]:
# Cell 15 — Step 2 (the "FT" of LP-FT): optimiser, learning-rate schedule and checkpoint resume.
adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "lorma_" in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "lorma_" not in n]   # projection
optimizer = torch.optim.AdamW([{"params": adapter_params, "lr": LR_ADAPTER}, {"params": head_params, "lr": LR_HEAD}],
                              weight_decay=WEIGHT_DECAY)

def epoch_pairs(epoch):
    order = TRAIN_PAIRS[:]
    random.Random(SEED + epoch).shuffle(order)                  # same batches after a resume
    return [order[s:s + PAIRS_PER_STEP] for s in range(0, len(order), PAIRS_PER_STEP)]

steps_per_epoch = len(epoch_pairs(0))
total_updates = EPOCHS * steps_per_epoch
warmup_updates = int(WARMUP_FRACTION * total_updates)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda u: (u + 1) / max(warmup_updates, 1) if u < warmup_updates
                                              else max(0.0, (total_updates - u) / max(total_updates - warmup_updates, 1)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

if not LATEST.exists():
    # Resume only from a run of this notebook with the same saved parameter names.
    for previous in sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/latest.pt", recursive=True)):
        if set(torch.load(previous, map_location="cpu", weights_only=False)["trainable"]) == set(SAVED_NAMES):
            shutil.copy(previous, LATEST)
            if Path(previous).with_name("best.pt").exists():
                shutil.copy(Path(previous).with_name("best.pt"), BEST)
            print("Copied checkpoints from", Path(previous).parent)
            break

state = {"epoch": 0, "step": 0, "best_f1": probe_dev["micro_f1@7"], "best_epoch": 0, "history": [], "checks": [],
         "train_seconds": 0.0}
if LATEST.exists():
    ckpt = torch.load(LATEST, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["trainable"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"]); scheduler.load_state_dict(ckpt["scheduler"]); scaler.load_state_dict(ckpt["scaler"])
    state = ckpt["state"]
    print(f"Resumed at epoch {state['epoch'] + 1}, step {state['step']} (best dev Micro-F1@7 {state['best_f1']:.4f})")
else:
    print("No checkpoint found — fine-tuning starts from the Phase-1 student weights + the trained projection.")
print(f"{steps_per_epoch:,} steps per epoch ({PAIRS_PER_STEP} pairs each), {total_updates:,} optimiser updates in total")

No checkpoint found — fine-tuning starts from the Phase-1 student weights + the trained projection.
1,327 steps per epoch (4 pairs each), 3,981 optimiser updates in total


In [16]:
# Cell 16 — Fine-tuning the student's LORMA + projection with InfoNCE + KD; frozen weights run without dropout and with
# gradient checkpointing. Quick dev checks during the epoch; full dev evaluation after every epoch; best.pt = best of 0…EPOCHS.
def save_latest():
    torch.save({"trainable": trainable_state(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}, LATEST)

def train_doc_vectors(docs):
    chunks, lengths = [], []
    for d in docs:
        cs = doc_chunks("train", d, TRAIN_SETTING)
        chunks += cs; lengths.append(len(cs))
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        raw = encoder(chunk_batch(chunks))
    return pool(model.chunk_vectors(raw.float()), lengths)

def dev_check():
    vecs = case_vectors("dev")
    result, _ = evaluate_vectors(vecs, "dev", DEV_QUERIES)
    return result, spread(vecs)

model.encoder.backbone.grad_checkpointing = GRAD_CHECKPOINTING
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
while state["epoch"] < EPOCHS:
    epoch = state["epoch"]
    batches = epoch_pairs(epoch)
    rng = random.Random(SEED * 1000 + epoch)
    for _ in range(state["step"]):                              # replay the hard-negative draws after a resume
        pair_batch(batches[_], rng)
    model.eval()                                                # dropout off everywhere; only LORMA + projection learn
    start_time, loss_sum, loss_count = time.time(), 0.0, 0
    for step in range(state["step"], len(batches)):
        pairs = batches[step]
        docs = pair_batch(pairs, rng)
        loss = contrastive_loss(train_doc_vectors(docs), docs, pairs)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(adapter_params + head_params, GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        loss_sum, loss_count = loss_sum + loss.item(), loss_count + 1
        state["step"] = step + 1
        if state["step"] % SAVE_EVERY_STEPS == 0:
            state["train_seconds"] += time.time() - start_time; start_time = time.time()
            save_latest()
            print(f"  epoch {epoch + 1} step {state['step']}/{len(batches)} | mean loss {loss_sum / loss_count:.4f} "
                  f"(last InfoNCE {LOSS_PARTS['ce']:.3f}, KD {LOSS_PARTS['kd']:.3f})", flush=True)
        if state["step"] % CHECK_EVERY_STEPS == 0:
            quick, sp = dev_check()
            state["checks"].append({"epoch": epoch + 1, "step": state["step"], "micro_f1@7": quick["micro_f1@7"], "spread": sp})
            warn = "  WARNING: case vectors nearly identical — collapse" if sp > 0.98 else ""
            print(f"    dev check: {describe(quick)} | mean pair cosine {sp:.3f}{warn}", flush=True)
            start_time = time.time()
    state["train_seconds"] += time.time() - start_time
    dev, sp = dev_check()
    state["history"].append({"epoch": epoch + 1, "train_loss": loss_sum / max(loss_count, 1), "dev": dev, "spread": sp})
    mark = ""
    if dev["micro_f1@7"] > state["best_f1"]:
        state.update(best_f1=dev["micro_f1@7"], best_epoch=epoch + 1)
        torch.save({"trainable": trainable_state(), "epoch": epoch + 1, "dev": dev}, BEST)
        mark = "  <- best"
    state["epoch"], state["step"] = epoch + 1, 0
    save_latest()
    print(f"Epoch {epoch + 1} | loss {loss_sum / max(loss_count, 1):.4f} | dev {describe(dev)} | pair cosine {sp:.3f}{mark}", flush=True)
model.encoder.backbone.grad_checkpointing = False
print(f"Best dev Micro-F1@7 {state['best_f1']:.4f} at epoch {state['best_epoch']} (0 = projection only) | "
      f"training time {state['train_seconds'] / 60:.1f} min")

  epoch 1 step 300/1327 | mean loss 1.9587 (last InfoNCE 0.167, KD 0.660)
  epoch 1 step 600/1327 | mean loss 1.9314 (last InfoNCE 0.726, KD 0.533)
    dev check: Micro-F1@7 0.2790 (P 0.3039, R 0.2580) | Recall@100 0.6855 | mean pair cosine 0.371
  epoch 1 step 900/1327 | mean loss 1.9052 (last InfoNCE 0.788, KD 0.556)
  epoch 1 step 1200/1327 | mean loss 1.8887 (last InfoNCE 1.030, KD 0.459)
    dev check: Micro-F1@7 0.2713 (P 0.2954, R 0.2508) | Recall@100 0.6958 | mean pair cosine 0.358
Epoch 1 | loss 1.8809 | dev Micro-F1@7 0.2802 (P 0.3051, R 0.2590) | Recall@100 0.6732 | pair cosine 0.415  <- best
  epoch 2 step 300/1327 | mean loss 1.6778 (last InfoNCE 0.451, KD 0.498)
  epoch 2 step 600/1327 | mean loss 1.6814 (last InfoNCE 0.574, KD 0.346)
    dev check: Micro-F1@7 0.2768 (P 0.3015, R 0.2559) | Recall@100 0.6824 | mean pair cosine 0.419
  epoch 2 step 900/1327 | mean loss 1.6744 (last InfoNCE 1.032, KD 0.468)
  epoch 2 step 1200/1327 | mean loss 1.6902 (last InfoNCE 0.301, KD 

In [17]:
# Cell 17 — Best model on dev: sections and pooling, score by number of relevant cases, queries without a hit, and how the
# model's hits overlap with TF-IDF's (the reference ranker).
best = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(best["trainable"], strict=False)
model.eval()
print(f"Best epoch {best['epoch']} (0 = projection only): {describe(best['dev'])}")

section_results = {}
for setting in SECTION_SETTINGS:
    feats, lengths, _ = raw_features("dev", setting)
    vecs = model.chunk_vectors(feats.to(DEVICE))
    for mode in ("mean", "max"):
        result, _ = evaluate_vectors(pool(vecs, lengths, mode), "dev", DEV_QUERIES)
        section_results[f"{setting[0]}_{setting[1]}_{mode}"] = result
        print(f"  {setting[0]:9s} {setting[1]:2d} chunks, {mode:4s} pooling: {describe(result)}")
BEST_SECTION = max(section_results, key=lambda k: section_results[k]["micro_f1@7"])
kind, k, mode = BEST_SECTION.split("_")
TEST_SETTING, TEST_POOLING = (kind, int(k)), mode
print(f"Best on dev: {BEST_SECTION} → used for the test predictions")

dev_vecs = case_vectors("dev", TEST_SETTING, TEST_POOLING)
dev_result, dev_ranked = evaluate_vectors(dev_vecs, "dev", DEV_QUERIES)
by_bucket = {}
for q, r in zip(DEV_QUERIES, dev_ranked):
    g = set(QUERIES["dev"][q]["rel"]); n = len(g)
    bucket = "1" if n == 1 else "2-3" if n <= 3 else "4-7" if n <= K else "8-15" if n <= 15 else "16+"
    by_bucket.setdefault(bucket, []).append((len(set(r[:K]) & g), n))
dev_by_n = {b: {"queries": len(v), "micro_p@7": round(sum(h for h, _ in v) / (K * len(v)), 4),
                "micro_r@7": round(sum(h for h, _ in v) / sum(n for _, n in v), 4)} for b, v in sorted(by_bucket.items())}
no_hit = [q for q, r in zip(DEV_QUERIES, dev_ranked) if not set(r[:K]) & set(QUERIES["dev"][q]["rel"])]
model_hits = {q: set(r[:K]) & set(QUERIES["dev"][q]["rel"]) for q, r in zip(DEV_QUERIES, dev_ranked)}
tfidf_hits = {q: set(HARD["tfidf_dev_top7"][q]) & set(QUERIES["dev"][q]["rel"]) for q in DEV_QUERIES}
overlap = {"model_hits": sum(len(v) for v in model_hits.values()), "tfidf_hits": sum(len(v) for v in tfidf_hits.values()),
           "both": sum(len(model_hits[q] & tfidf_hits[q]) for q in DEV_QUERIES),
           "union": sum(len(model_hits[q] | tfidf_hits[q]) for q in DEV_QUERIES)}
print("dev by number of relevant cases:", json.dumps(dev_by_n))
print(f"queries without any hit in the top 7: {len(no_hit)} of {len(DEV_QUERIES)}")
print("hits shared with TF-IDF:", overlap)

Best epoch 1 (0 = projection only): Micro-F1@7 0.2802 (P 0.3051, R 0.2590) | Recall@100 0.6732
  first      1 chunks, mean pooling: Micro-F1@7 0.1234 (P 0.1344, R 0.1141) | Recall@100 0.5077
  first      1 chunks, max  pooling: Micro-F1@7 0.1234 (P 0.1344, R 0.1141) | Recall@100 0.5077
  first      4 chunks, mean pooling: Micro-F1@7 0.2301 (P 0.2506, R 0.2127) | Recall@100 0.6608
  first      4 chunks, max  pooling: Micro-F1@7 0.1923 (P 0.2094, R 0.1778) | Recall@100 0.5992
  last       4 chunks, mean pooling: Micro-F1@7 0.2246 (P 0.2446, R 0.2076) | Recall@100 0.6321
  last       4 chunks, max  pooling: Micro-F1@7 0.1712 (P 0.1864, R 0.1583) | Recall@100 0.5396
  firstlast  8 chunks, mean pooling: Micro-F1@7 0.2802 (P 0.3051, R 0.2590) | Recall@100 0.6732
  firstlast  8 chunks, max  pooling: Micro-F1@7 0.1879 (P 0.2046, R 0.1737) | Recall@100 0.5488
  firstlast 16 chunks, mean pooling: Micro-F1@7 0.2746 (P 0.2990, R 0.2539) | Recall@100 0.6989
  firstlast 16 chunks, max  pooling: Micr

In [18]:
# Cell 18 — Did KD keep the quality, and at what cost? Teacher (rebuilt from S1's best.pt) vs student on the same COST_DOCS
# dev documents, one GPU each: encoding time and peak memory; plus parameters, dev Micro-F1@7 and top-7 agreement.
t_ckpt = sorted(glob.glob(f"/kaggle/input/**/{TEACHER_RUN}/checkpoints/best.pt", recursive=True))
assert t_ckpt, "S1's checkpoints/best.pt is missing from Phase7_S1_Dataset"
teacher_model = Retriever(make_lorma_backbone(A_CONFIG, PHASE1_STATE), A_CONFIG["d_model"], PROJ_DIM).to(DEVICE).eval()
t_state = torch.load(t_ckpt[0], map_location="cpu", weights_only=False)["trainable"]
missing = set(t_state) - set(teacher_model.state_dict())
assert not missing, f"teacher checkpoint has unknown tensors: {list(missing)[:3]}"
teacher_model.load_state_dict(t_state, strict=False)

@torch.no_grad()
def timed_vectors(retriever, docs, setting, pooling):
    """Case vectors of `docs` on one GPU, with wall time and peak memory."""
    chunks, lengths = [], []
    for d in docs:
        cs = doc_chunks("dev", d, setting)
        chunks += cs; lengths.append(len(cs))
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    start = time.time()
    raw = []
    for s in range(0, len(chunks), EVAL_CHUNK_BATCH // 2):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            raw.append(retriever.encoder(chunk_batch(chunks[s:s + EVAL_CHUNK_BATCH // 2])).float())
    vecs = pool(retriever.chunk_vectors(torch.cat(raw)), lengths, pooling)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else None
    return vecs, time.time() - start, peak, len(chunks)

cost_docs = list(range(min(COST_DOCS, len(DOC_TEXT["dev"]))))
timed_vectors(model, cost_docs[:5], TEST_SETTING, TEST_POOLING)                # warm-up
s_vecs, s_time, s_peak, n_chunks = timed_vectors(model, cost_docs, TEST_SETTING, TEST_POOLING)
t_vecs, t_time, t_peak, _ = timed_vectors(teacher_model, cost_docs, TEST_SETTING, TEST_POOLING)   # same chunks: fair timing
t_own, _, _, _ = timed_vectors(teacher_model, cost_docs, tuple(TEACHER["setting"]), TEACHER["pooling"])
t_match = float((F.normalize(t_own, dim=-1) * T_VECS["dev"][cost_docs]).sum(-1).mean())   # rebuilt teacher = S1?
teacher_top = dict(zip(DEV_QUERIES, [r[:K] for r in teacher_dev_ranked]))
agreement = float(np.mean([len(set(teacher_top[q]) & set(r[:K])) / K for q, r in zip(DEV_QUERIES, dev_ranked)]))
n_teacher = sum(p.numel() for p in teacher_model.parameters())
kd_report = {
    "teacher": {"params": n_teacher, "dev_micro_f1@7": teacher_dev["micro_f1@7"], "seconds_100_docs": round(t_time, 2),
                "peak_gpu_gb": round(t_peak, 2) if t_peak else None},
    "student": {"params": n_total, "dev_micro_f1@7": dev_result["micro_f1@7"], "seconds_100_docs": round(s_time, 2),
                "peak_gpu_gb": round(s_peak, 2) if s_peak else None},
    "student_share_of_teacher_params": round(n_total / n_teacher, 3),
    "quality_kept": round(dev_result["micro_f1@7"] / max(teacher_dev["micro_f1@7"], 1e-9), 3),
    "speedup": round(t_time / max(s_time, 1e-9), 2), "top7_agreement_with_teacher": round(agreement, 3),
    "chunks_encoded": n_chunks, "rebuilt_teacher_matches_saved_vectors_cosine": round(t_match, 4),
}
print(json.dumps(kd_report, indent=2))
assert t_match > 0.98, "the rebuilt teacher does not reproduce S1's saved vectors"
del teacher_model
torch.cuda.empty_cache()

{
  "teacher": {
    "params": 126749440,
    "dev_micro_f1@7": 0.269,
    "seconds_100_docs": 7.04,
    "peak_gpu_gb": 1.54
  },
  "student": {
    "params": 83558656,
    "dev_micro_f1@7": 0.2802,
    "seconds_100_docs": 3.54,
    "peak_gpu_gb": 1.54
  },
  "student_share_of_teacher_params": 0.659,
  "quality_kept": 1.042,
  "speedup": 1.99,
  "top7_agreement_with_teacher": 0.494,
  "chunks_encoded": 647,
  "rebuilt_teacher_matches_saved_vectors_cosine": 1.0
}


In [19]:
# Cell 19 — Predict the test queries (query removed from its own ranking), validate the format, build submission.zip.
# prediction.json = {query id: [7 unique candidate ids from the test pool, most relevant first]} for every test query.
test_vecs = case_vectors("test", TEST_SETTING, TEST_POOLING)
rows = torch.tensor([DOC_INDEX["test"][QUERIES["test"][q]["key"]] for q in TEST_QUERIES], device=test_vecs.device)
test_ranked = rank_queries((test_vecs[rows] @ test_vecs.T).cpu(), "test", TEST_QUERIES, depth=K)
prediction = {q: r[:K] for q, r in zip(TEST_QUERIES, test_ranked)}
pool_set = set(POOL_IDS["test"])
assert set(prediction) == set(QUERIES["test"]), "every test query needs a prediction"
assert all(len(v) == K and len(set(v)) == K and set(v) <= pool_set and q not in v for q, v in prediction.items())
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"))
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
counts = Counter(c for v in prediction.values() for c in v)
print(f"submission.zip: {len(prediction)} test queries × {K} candidates | distinct candidates used {len(counts)} | "
      f"most repeated candidate appears for {counts.most_common(1)[0][1]} queries")

submission.zip: 237 test queries × 7 candidates | distinct candidates used 966 | most repeated candidate appears for 8 queries


In [20]:
# Cell 20 — Final student case vectors of every train, dev and test document (TEST_SETTING / TEST_POOLING), with
# document keys (for comparisons and ensembling with the other runs).
start_time = time.time()
teacher = {"setting": list(TEST_SETTING), "pooling": TEST_POOLING, "dim": PROJ_DIM, "dev_result": dev_result}
for split in ("train", "dev", "test"):
    vecs = case_vectors(split, TEST_SETTING, TEST_POOLING) if split != "dev" else dev_vecs
    teacher[split] = {"keys": list(DOC_INDEX[split]), "vectors": vecs.half().cpu()}
torch.save(teacher, OUT_DIR / "student_vectors.pt")
check = teacher["dev"]["vectors"].float().to(DEVICE)
again, _ = evaluate_vectors(check, "dev", DEV_QUERIES)
assert abs(again["micro_f1@7"] - dev_result["micro_f1@7"]) < 0.01, "saved student vectors do not reproduce the dev score"
print(f"Student vectors saved to {OUT_DIR / 'student_vectors.pt'} in {time.time() - start_time:.0f} s | dev check {describe(again)}")

Student vectors saved to /kaggle/working/P7_S4_KD_S1Teacher_A6LayerStudent/outputs/student_vectors.pt in 135 s | dev check Micro-F1@7 0.2802 (P 0.3051, R 0.2590) | Recall@100 0.6732


In [21]:
# Cell 21 — Run summary for the README, the experiment log and results/submissions.csv.
summary = {
    "run": RUN_NAME, "backbone": "Architecture A, 6 of 12 Phase-1 layers (student); teacher = P7_S1 (12-layer A + LORMA)",
    "technique": "knowledge distillation (listwise KL on similarity scores + teacher-mined hard negatives) into a 6-layer A student with LORMA, LP-FT",
    "kd": kd_report, "student_layers": list(STUDENT_LAYERS), "kd_weight": KD_WEIGHT, "kd_temperature": KD_TEMPERATURE,
    "lorma_rank": LORMA_RANK, "trainable_params": n_trainable, "total_params": n_total,
    "train_queries": len(TRAIN_QUERIES), "train_pairs": len(TRAIN_PAIRS), "dev_queries": len(DEV_QUERIES),
    "tfidf_dev_reference": HARD["tfidf_dev"], "frozen_transfer": transfer,
    "probe": {"best_epoch": PROBE_EPOCH, "dev": probe_dev, "history": probe_history},
    "epochs": state["epoch"], "best_epoch": state["best_epoch"], "history": state["history"], "checks": state["checks"],
    "train_minutes": round(state["train_seconds"] / 60, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "sections": {k: v["micro_f1@7"] for k, v in section_results.items()}, "test_setting": [*TEST_SETTING, TEST_POOLING],
    "dev": dev_result, "dev_by_n_relevant": dev_by_n, "dev_queries_without_hit": len(no_hit), "hits_vs_tfidf": overlap,
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps({k: v for k, v in summary.items() if k not in ("frozen_transfer", "probe", "history", "checks")}, indent=2))

{
  "run": "P7_S4_KD_S1Teacher_A6LayerStudent",
  "backbone": "Architecture A, 6 of 12 Phase-1 layers (student); teacher = P7_S1 (12-layer A + LORMA)",
  "technique": "knowledge distillation (listwise KL on similarity scores + teacher-mined hard negatives) into a 6-layer A student with LORMA, LP-FT",
  "kd": {
    "teacher": {
      "params": 126749440,
      "dev_micro_f1@7": 0.269,
      "seconds_100_docs": 7.04,
      "peak_gpu_gb": 1.54
    },
    "student": {
      "params": 83558656,
      "dev_micro_f1@7": 0.2802,
      "seconds_100_docs": 3.54,
      "peak_gpu_gb": 1.54
    },
    "student_share_of_teacher_params": 0.659,
    "quality_kept": 1.042,
    "speedup": 1.99,
    "top7_agreement_with_teacher": 0.494,
    "chunks_encoded": 647,
    "rebuilt_teacher_matches_saved_vectors_cosine": 1.0
  },
  "student_layers": [
    0,
    2,
    4,
    6,
    8,
    10
  ],
  "kd_weight": 1.0,
  "kd_temperature": 2.0,
  "lorma_rank": 8,
  "trainable_params": 1057024,
  "total_params": 83